In [0]:

# Step 1: Read incremental Silver data

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

df_silver = spark.table(silver_table)

print("Incremental Silver records:", df_silver.count())
print("Total columns:", len(df_silver.columns))

display(df_silver.limit(5))


In [0]:

# Step 2: Calculate transaction revenue

from pyspark.sql.functions import col, round as spark_round

df_gold_transactions = (
    df_silver
    .withColumn(
        "QuantityNumeric",
        col("Quantity").cast("decimal(18,2)")
    )
    .withColumn(
        "UnitPriceNumeric",
        col("UnitPrice").cast("decimal(18,2)")
    )
    .withColumn(
        "LineTotal",
        spark_round(
            col("QuantityNumeric") * col("UnitPriceNumeric"),
            2
        )
    )
)

print("Total transactions:", df_gold_transactions.count())

display(
    df_gold_transactions.select(
        "InvoiceNo",
        "StockCode",
        "QuantityNumeric",
        "UnitPriceNumeric",
        "LineTotal"
    ).limit(10)
)


In [0]:

from pyspark.sql import functions as F

# Step 3: Create the incremental Gold revenue table

gold_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_gold_revenue_incremental"
)

# Calculate revenue from the 200 Silver records
df_gold_revenue = df_gold_transactions.agg(
    F.count("*").alias("TotalTransactions"),
    F.round(
        F.sum("LineTotal"), 2
    ).alias("TotalNetRevenue")
)

print("Calculated Gold revenue:")
display(df_gold_revenue)

# Create the Gold table if it does not exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_table} (
    TotalTransactions BIGINT,
    TotalNetRevenue DECIMAL(38,2)
)
USING DELTA
""")

# Insert the initial revenue only if the table is empty
if spark.table(gold_table).count() == 0:

    df_gold_revenue.selectExpr(
        "CAST(TotalTransactions AS BIGINT) AS TotalTransactions",
        "CAST(TotalNetRevenue AS DECIMAL(38,2)) AS TotalNetRevenue"
    ).write.format("delta").mode("append").saveAsTable(gold_table)

    print("Gold table initialized successfully!")

else:
    print("Gold table already contains data. Skipping initialization.")

# Verify the Gold table
print("Final Gold table:")
display(spark.table(gold_table))


In [0]:

gold_table = "dbw_retail_dataeng.retail_part2_dev.retail_gold_revenue_incremental"

print("Gold table exists:", spark.catalog.tableExists(gold_table))


In [0]:

# Step 4: Inspect Silver Delta table versions

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

df_history = spark.sql(
    f"DESCRIBE HISTORY {silver_table}"
)

display(
    df_history.select(
        "version",
        "timestamp",
        "operation"
    ).orderBy("version", ascending=False)
    .limit(10)
)


In [0]:

# Check Delta Change Data Feed configuration

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

df_properties = spark.sql(
    f"SHOW TBLPROPERTIES {silver_table}"
)

display(
    df_properties.filter(
        "key = 'delta.enableChangeDataFeed'"
    )
)


In [0]:

# Enable Delta Change Data Feed on the Silver table

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

spark.sql(f"""
ALTER TABLE {silver_table}
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
)
""")

print("Delta Change Data Feed enabled successfully!")


In [0]:

# Step 5: Check the latest Silver Delta version

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

history = spark.sql(
    f"DESCRIBE HISTORY {silver_table}"
)

latest = history.orderBy("version", ascending=False).first()

print("Latest Silver Version:", latest["version"])
print("Latest Operation:", latest["operation"])
print("Timestamp:", latest["timestamp"])

# Verify CDF is enabled
properties = spark.sql(
    f"SHOW TBLPROPERTIES {silver_table}"
)

display(
    properties.filter(
        "key = 'delta.enableChangeDataFeed'"
    )
)


In [0]:

# Check the latest Silver Delta version

history = spark.sql("""
DESCRIBE HISTORY
dbw_retail_dataeng.retail_part2_dev.retail_merge_test
""")

display(
    history.select(
        "version",
        "operation",
        "timestamp"
    ).orderBy("version", ascending=False).limit(3)
)


In [0]:

# Step 6: Create Gold incremental processing checkpoint

checkpoint_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_gold_checkpoint"
)

# Create checkpoint table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {checkpoint_table} (
    pipeline_name STRING,
    last_processed_version BIGINT,
    updated_at TIMESTAMP
)
USING DELTA
""")

# Initialize checkpoint only if no checkpoint exists
checkpoint_exists = spark.sql(f"""
SELECT COUNT(*) AS total
FROM {checkpoint_table}
WHERE pipeline_name = 'silver_to_gold_revenue'
""").first()["total"]

if checkpoint_exists == 0:

    spark.sql(f"""
    INSERT INTO {checkpoint_table}
    VALUES (
        'silver_to_gold_revenue',
        7,
        current_timestamp()
    )
    """)

    print("Gold checkpoint initialized at Silver version 7")

else:
    print("Existing Gold checkpoint preserved")

# Verify checkpoint
display(spark.sql(f"""
SELECT *
FROM {checkpoint_table}
WHERE pipeline_name = 'silver_to_gold_revenue'
"""))


In [0]:

# Step 7: Read new Silver changes using Delta CDF

from pyspark.sql import functions as F

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

checkpoint_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_gold_checkpoint"
)

# Read the last processed Silver version
checkpoint_version = spark.sql(f"""
SELECT last_processed_version
FROM {checkpoint_table}
WHERE pipeline_name = 'silver_to_gold_revenue'
""").first()["last_processed_version"]

# Find the latest available Silver version
latest_version = spark.sql(
    f"DESCRIBE HISTORY {silver_table}"
).select(
    F.max("version").alias("latest_version")
).first()["latest_version"]

print("Last processed version:", checkpoint_version)
print("Latest Silver version:", latest_version)

if latest_version > checkpoint_version:

    df_changes = (
        spark.read
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", checkpoint_version + 1)
        .option("endingVersion", latest_version)
        .table(silver_table)
    )

    display(
        df_changes.select(
            "test_record_id",
            "Quantity",
            "UnitPrice",
            "_change_type",
            "_commit_version"
        )
    )

else:
    print("No new Silver changes to process.")


In [0]:

from pyspark.sql import functions as F

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

test_id = "gold_cdf_test_001"

# Check whether the test record already exists
existing = (
    spark.table(silver_table)
    .filter(F.col("test_record_id") == test_id)
    .count()
)

if existing == 0:

    test_record = spark.createDataFrame(
        [(
            "GOLD-TEST-001",
            "TEST-ITEM-001",
            "Incremental Gold Test Item",
            "2",
            "2026-10-23 12:00:00",
            "15.50",
            "TEST-CUSTOMER",
            "Canada",
            "gold_cdf_test",
            test_id
        )],
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country",
            "source_file",
            "test_record_id"
        ]
    )

    (
        test_record.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table)
    )

    print("Test transaction inserted successfully!")

else:
    print("Test transaction already exists. Skipping insertion.")

print(
    "Current Silver records:",
    spark.table(silver_table).count()
)


In [0]:

from pyspark.sql import functions as F

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

checkpoint_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_gold_checkpoint"
)

# Get last processed version
checkpoint_version = spark.sql(f"""
    SELECT last_processed_version
    FROM {checkpoint_table}
    WHERE pipeline_name = 'silver_to_gold_revenue'
""").first()["last_processed_version"]

# Get latest Silver version
latest_version = spark.sql(
    f"DESCRIBE HISTORY {silver_table}"
).agg(
    F.max("version").alias("latest")
).first()["latest"]

print("Checkpoint version:", checkpoint_version)
print("Latest Silver version:", latest_version)

# Read changes since the checkpoint
if latest_version > checkpoint_version:
    df_changes = (
        spark.read
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", checkpoint_version + 1)
        .option("endingVersion", latest_version)
        .table(silver_table)
    )

    display(
        df_changes.select(
            "test_record_id",
            "Quantity",
            "UnitPrice",
            "_change_type",
            "_commit_version"
        )
    )
else:
    print("No new Silver changes.")



In [0]:

from pyspark.sql import functions as F

# Calculate the revenue effect of each CDF change
df_revenue_changes = (
    df_changes
    .withColumn(
        "LineTotal",
        F.col("Quantity").cast("decimal(18,2)") *
        F.col("UnitPrice").cast("decimal(18,2)")
    )
    .withColumn(
        "RevenueAdjustment",
        F.when(
            F.col("_change_type").isin("insert", "update_postimage"),
            F.col("LineTotal")
        )
        .when(
            F.col("_change_type").isin("delete", "update_preimage"),
            -F.col("LineTotal")
        )
        .otherwise(F.lit(0))
    )
    .withColumn(
        "TransactionAdjustment",
        F.when(
            F.col("_change_type") == "insert", 1
        )
        .when(
            F.col("_change_type") == "delete", -1
        )
        .otherwise(0)
    )
)

# Calculate total changes
df_gold_adjustment = df_revenue_changes.agg(
    F.sum("TransactionAdjustment").alias("NewTransactions"),
    F.round(
        F.sum("RevenueAdjustment"), 2
    ).alias("RevenueIncrease")
)

display(df_gold_adjustment)


In [0]:

from decimal import Decimal

# Read the calculated CDF adjustment
adjustment = df_gold_adjustment.first()

new_transactions = adjustment["NewTransactions"]
revenue_increase = adjustment["RevenueIncrease"]

print("New transactions:", new_transactions)
print("Revenue adjustment:", revenue_increase)

# Validate the controlled test
assert new_transactions == 1, "Unexpected transaction count"
assert revenue_increase == Decimal("31.00"), "Unexpected revenue adjustment"

# Verify checkpoint has not advanced
current_checkpoint = spark.sql(f"""
    SELECT last_processed_version
    FROM {checkpoint_table}
    WHERE pipeline_name = 'silver_to_gold_revenue'
""").first()["last_processed_version"]

assert current_checkpoint == 7, "Checkpoint changed unexpectedly"

print("PASS: Gold adjustment validated")
print("PASS: Checkpoint remains at version 7")


In [0]:

# Step 8: Add an incremental processing checkpoint to Gold

gold_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_gold_revenue_incremental"
)

# Add the checkpoint column only if it is missing
existing_columns = spark.table(gold_table).columns

if "last_applied_version" not in existing_columns:
    spark.sql(f"""
        ALTER TABLE {gold_table}
        ADD COLUMNS (
            last_applied_version BIGINT
        )
    """)

    print("Checkpoint column added.")
else:
    print("Checkpoint column already exists.")

# Initialize the version for our validated Gold baseline
spark.sql(f"""
    UPDATE {gold_table}
    SET last_applied_version = 7
    WHERE last_applied_version IS NULL
""")

# Verify the Gold state
display(
    spark.table(gold_table).select(
        "TotalTransactions",
        "TotalNetRevenue",
        "last_applied_version"
    )
)


In [0]:

from pyspark.sql import functions as F

gold_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_gold_revenue_incremental"
)

# Use the Gold table's own checkpoint
gold_state = spark.table(gold_table).first()
gold_version = gold_state["last_applied_version"]

# Never apply changes already reflected in Gold
if latest_version <= gold_version:
    print("SKIPPED: Gold is already up to date.")

else:
    # Read exactly the unapplied Silver changes
    df_pending = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", gold_version + 1)
        .option("endingVersion", latest_version)
        .table(silver_table)
    )

    signed_changes = (
        df_pending
        .withColumn(
            "amount",
            F.col("Quantity").cast("decimal(18,2)") *
            F.col("UnitPrice").cast("decimal(18,2)")
        )
        .withColumn(
            "revenue_delta",
            F.when(
                F.col("_change_type").isin(
                    "insert", "update_postimage"
                ),
                F.col("amount")
            ).otherwise(-F.col("amount"))
        )
        .withColumn(
            "count_delta",
            F.when(F.col("_change_type") == "insert", 1)
            .when(F.col("_change_type") == "delete", -1)
            .otherwise(0)
        )
    )

    totals = signed_changes.agg(
        F.sum("count_delta").alias("count_delta"),
        F.sum("revenue_delta").alias("revenue_delta")
    ).first()

    count_delta = totals["count_delta"] or 0
    revenue_delta = totals["revenue_delta"] or 0

    # Stage the adjustment as one source row
    spark.createDataFrame(
        [(int(count_delta), str(revenue_delta), int(latest_version))],
        ["count_delta", "revenue_delta_text", "target_version"]
    ).createOrReplaceTempView("gold_incremental_adjustment")

    spark.sql(f"""
        MERGE INTO {gold_table} AS target
        USING (
            SELECT
                count_delta,
                CAST(revenue_delta_text AS DECIMAL(38,2))
                    AS revenue_delta,
                target_version
            FROM gold_incremental_adjustment
        ) AS source
        ON target.last_applied_version < source.target_version
        WHEN MATCHED THEN UPDATE SET
            target.TotalTransactions =
                target.TotalTransactions + source.count_delta,
            target.TotalNetRevenue =
                target.TotalNetRevenue + source.revenue_delta,
            target.last_applied_version = source.target_version
    """)

    print("Gold incremental MERGE completed.")

display(spark.table(gold_table))


In [0]:

# Step 9: Validate Gold and synchronize checkpoint

gold_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_gold_revenue_incremental"
)

checkpoint_table = (
    "dbw_retail_dataeng.retail_part2_dev."
    "retail_gold_checkpoint"
)

# Read Gold's authoritative processing version
gold_state = spark.table(gold_table).first()

gold_version = gold_state["last_applied_version"]
gold_count = gold_state["TotalTransactions"]
gold_revenue = gold_state["TotalNetRevenue"]

# Validate the expected test result
assert gold_count == 201, "Unexpected Gold transaction count"
assert gold_version >= 8, "Gold version was not advanced"

print("Gold transactions:", gold_count)
print("Gold revenue:", gold_revenue)
print("Gold applied version:", gold_version)

# Synchronize the separate tracking checkpoint
spark.sql(f"""
    UPDATE {checkpoint_table}
    SET
        last_processed_version = {gold_version},
        updated_at = current_timestamp()
    WHERE pipeline_name = 'silver_to_gold_revenue'
      AND last_processed_version < {gold_version}
""")

print("Checkpoint synchronized.")

# Check whether any new Silver changes remain
current_silver_version = spark.sql(f"""
    DESCRIBE HISTORY {silver_table}
""").selectExpr("MAX(version) AS version").first()["version"]

if current_silver_version <= gold_version:
    print("PASS: No unprocessed Silver changes.")
else:
    print("New Silver changes are waiting for processing.")

display(spark.table(checkpoint_table))

